# Team Construction — Iteration Notebook

Polish how the orchestrator picks specialists and decomposes sub-questions.

**Edit surfaces:**
- Non-engineer: `skills/workflow/team_construction.md`, `config/pillars/<pillar>.yaml`, `skills/domain/*.md`.
- Engineer: `orchestrator/orchestrator.py` (see `_select_team`, `_split_sub_questions`).

**Workflow:** edit a file above → Run All → read Cell 5's rendered plan → repeat. Cell 6's raw JSON dump is the regression signal — commit the notebook with output cells populated so future diffs are meaningful.

In [ ]:
# ═══════════════ KNOBS ═══════════════
FIXTURE = "basic_case"
REGENERATE = False          # True = write current values back to fixtures/team_construction/<FIXTURE>.json
MODEL = "gpt-4.1"
# ═════════════════════════════════════

import json
import os
import sys
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

from dotenv import load_dotenv
load_dotenv()

# Repo root on sys.path so the project modules import cleanly.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from agents.session_registry import SessionRegistry
from config.pillar_loader import PillarLoader
from datalayer.catalog import DataCatalog
from datalayer.gateway import LocalDataGateway
from datalayer.generator import DataGenerator
from llm.firewall_stack import FirewallStack
from llm.factory import build_llm
from logger.event_logger import EventLogger
from orchestrator.orchestrator import Orchestrator
from skills.domain.loader import list_domain_skills
from tools.data_tools import init_tools

FIXTURE_PATH = PROJECT_ROOT / "notebooks" / "fixtures" / "team_construction" / f"{FIXTURE}.json"
print(f"Fixture: {FIXTURE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Regenerate: {REGENERATE}")

In [ ]:
# Logger + firewall + LLM
logger = EventLogger(session_id="polish-team-construction")
firewall = FirewallStack(logger=logger)
llm = build_llm(MODEL, firewall)

# Load the pillar YAML BEFORE we know which pillar we want, so we can re-use
# PillarLoader after the fixture is loaded in Cell 4.
pillar_loader = PillarLoader(pillar_dir=str(PROJECT_ROOT / "config" / "pillars"))

# Gateway: CSV-first, generator fallback (same logic as main.py).
_DATA_TABLES_DIR = PROJECT_ROOT / "data_tables"
csv_gateway = LocalDataGateway.from_case_folders(str(_DATA_TABLES_DIR))
if csv_gateway.list_case_ids():
    gateway = csv_gateway
    print(f"Data source: csv ({len(csv_gateway.list_case_ids())} cases)")
else:
    gen = DataGenerator(
        profile_dir=str(PROJECT_ROOT / "config" / "data_profiles"),
        seed=42, cases=50,
    )
    gen.load_profiles()
    tables_raw = gen.generate_all()
    gateway = LocalDataGateway.from_generated(tables_raw)
    print(f"Data source: generator ({len(gateway.list_case_ids())} cases)")

catalog = DataCatalog(profile_dir=str(PROJECT_ROOT / "config" / "data_profiles"))
init_tools(gateway, catalog, logger=logger)

registry = SessionRegistry()
print(f"Available case IDs (first 5): {gateway.list_case_ids()[:5]}")

In [ ]:
if REGENERATE:
    current = {
        "question": "Is this customer's credit risk acceptable?",
        "pillar": "credit_risk",
        "available_specialists": None,
        "active_specialists": [],
        "case_id": gateway.list_case_ids()[0],
        "notes": f"Regenerated from case {gateway.list_case_ids()[0]}.",
    }
    FIXTURE_PATH.write_text(json.dumps(current, indent=2) + "\n")
    fixture = current
    print(f"Wrote fixture: {FIXTURE_PATH.relative_to(PROJECT_ROOT)}")
else:
    fixture = json.loads(FIXTURE_PATH.read_text())
    print(f"Loaded fixture: {FIXTURE_PATH.relative_to(PROJECT_ROOT)}")

# Pick a usable case_id: the fixture's preferred value, or the first available if it's missing.
available = gateway.list_case_ids()
case_id = fixture["case_id"] if fixture["case_id"] in available else available[0]
if case_id != fixture["case_id"]:
    print(f"  (fixture case '{fixture['case_id']}' not available; using '{case_id}')")
gateway.set_case(case_id)

pillar_yaml = pillar_loader.load(fixture["pillar"]) or {}
print(f"Pillar: {fixture['pillar']} | Case: {case_id}")
print(f"Question: {fixture['question']}")

In [ ]:
from IPython.display import Markdown, display
from skills.domain.loader import load_domain_skill

orchestrator = Orchestrator(
    llm, logger, registry, fixture["pillar"],
    pillar_config=pillar_yaml, catalog=catalog,
)

available_specialists = fixture["available_specialists"] or list_domain_skills()
active_specialists = fixture["active_specialists"]

plan = await orchestrator.plan_team(
    question=fixture["question"],
    available_specialists=available_specialists,
    active_specialists=active_specialists,
)

# Render
selected = [p.specialist for p in plan]
not_picked = [s for s in available_specialists if s not in selected]

lines = [f"### Root question\n\n{fixture['question']}\n"]
lines.append(f"**Pillar:** `{fixture['pillar']}` — **Case:** `{case_id}`\n")
lines.append(f"**Available specialists:** {', '.join(available_specialists)}\n")
lines.append(f"**Selected ({len(selected)}):**")
for a in plan:
    skill = load_domain_skill(a.specialist)
    tables = ", ".join(skill.data_hints) if skill and skill.data_hints else "(no tables)"
    lines.append(f"- **{a.specialist}** — tables: {tables}")
    lines.append(f"  - sub-question: _{a.sub_question}_")
if not_picked:
    lines.append(f"\n**Not picked:** {', '.join(not_picked)}")
display(Markdown("\n".join(lines)))

In [ ]:
# This cell's output is the regression signal. Commit the notebook with this
# output populated; diff against future runs to see what an edit changed.
print(json.dumps([p.model_dump() for p in plan], indent=2))